# CS-781 Phase 4: Exp 2 — Layer-Wise Adaptation (Model 3 GPT-NeoX)Phase 4 Court Judgment Prediction submission.

In [ ]:
import os, json, glob, zipfile, shutil, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, precision_score, recall_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

HF_TOKEN = 'YOUR_HF_TOKEN'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TARGET_REPO = 'U-241111/cs781-submissions'

# 1. Download Phase 4 Dataset
os.makedirs('/kaggle/working/data_p4', exist_ok=True)
p4_files = [
    'Phase_4_CJPE/Train/train.json',
    'Phase_4_CJPE/Train/dev.json',
    'Phase_4_CJPE/Test/test.json'
]
for f in p4_files:
    fname = os.path.basename(f)
    dest = f'/kaggle/working/data_p4/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        loc = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(loc, dest)
print('Downloaded all Phase 4 files successfully!')


In [ ]:
# 2. Locate Checkpoint and Initialize Tokenizer
model_ckpt_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        model_ckpt_dir = root
        break
print('Model Checkpoint Path:', model_ckpt_dir, os.listdir(model_ckpt_dir))

tok = AutoTokenizer.from_pretrained(model_ckpt_dir)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'

# 3. Head + Tail Tokenization Dataset (Minimal Sufficient Statistic)
with open('/kaggle/working/data_p4/train.json') as f: train_data = json.load(f)
with open('/kaggle/working/data_p4/dev.json') as f: dev_data = json.load(f)
with open('/kaggle/working/data_p4/test.json') as f: test_data = json.load(f)

print(f'Loaded: Train={len(train_data)}, Dev={len(dev_data)}, Test={len(test_data)}')

def make_head_tail_tokens(text, tokenizer, max_head=256, max_tail=766):
    words = text.split()
    if len(words) > 1000:
        head_text = ' '.join(words[:350])
        tail_text = ' '.join(words[-1000:])
    else:
        head_text = text
        tail_text = ''
        
    head_ids = tokenizer.encode(f'[CASE_PREAMBLE] {head_text}', add_special_tokens=False)[:max_head]
    if tail_text:
        tail_ids = tokenizer.encode(f' [JUDICIAL_ANALYSIS] {tail_text}', add_special_tokens=False)[-max_tail:]
        combined = head_ids + tail_ids
    else:
        combined = head_ids
    return combined

class CJPEDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=1024, is_test=False):
        self.samples = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        tokens = make_head_tail_tokens(item['text'], self.tokenizer, max_head=256, max_tail=self.max_length - 258)
        
        # Left pad to max_length
        pad_len = max(0, self.max_length - len(tokens))
        input_ids = [self.tokenizer.pad_token_id] * pad_len + tokens[:self.max_length]
        attn_mask = [0] * pad_len + [1] * min(len(tokens), self.max_length)
        
        res = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attn_mask, dtype=torch.long),
            'id': item['id']
        }
        if not self.is_test:
            res['label'] = torch.tensor(int(item['label']), dtype=torch.long)
        return res


In [ ]:
# 4. Initialize Model 3 (GPT-NeoX) with Layer-Wise Adaptation
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt_dir,
    num_labels=2,
    id2label={0: 'Dismissed', 1: 'Allowed'},
    label2id={'Dismissed': 0, 'Allowed': 1}
)
model.config.pad_token_id = tok.pad_token_id

# Freeze lower layers, tune top 4 transformer blocks + score head
for name, p in model.named_parameters():
    p.requires_grad = False
    if any(f'layers.{i}.' in name for i in range(8, 12)) or 'score' in name or 'classifier' in name:
        p.requires_grad = True

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Layer-Wise Model: Trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

train_loader = DataLoader(CJPEDataset(train_data, tok, max_length=1024), batch_size=8, shuffle=True)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-5, weight_decay=0.01)
epochs = 3
grad_accum = 4
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

print(f'Starting training for {epochs} epochs...')
model.train()
for epoch in range(epochs):
    total_loss = 0
    t0 = time.time()
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        loss = criterion(logits, labels) / grad_accum
        loss.backward()
        
        if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
    print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s')


In [ ]:
SUB_ZIP_NAME = 'submission_phase4_exp2_layerwise.zip'# 5. Evaluation, Optimal Threshold Sweeping, and Test Submission
val_loader = DataLoader(CJPEDataset(dev_data, tok, max_length=1024), batch_size=16, shuffle=False)

model.eval()
val_preds_probs = []
val_trues = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        val_preds_probs.extend(probs)
        val_trues.extend(labels.cpu().numpy())

val_preds_probs = np.array(val_preds_probs)
val_trues = np.array(val_trues)

# Grid search optimal decision threshold on Dev Macro-F1
best_thresh = 0.50
best_macro_f1 = 0.0
for thresh in np.arange(0.30, 0.70, 0.005):
    preds = (val_preds_probs >= thresh).astype(int)
    mf1 = f1_score(val_trues, preds, average='macro', labels=[0, 1], zero_division=0)
    if mf1 > best_macro_f1:
        best_macro_f1 = mf1
        best_thresh = thresh

best_preds = (val_preds_probs >= best_thresh).astype(int)
val_prec = precision_score(val_trues, best_preds, average='binary', pos_label=1, zero_division=0)
val_rec = recall_score(val_trues, best_preds, average='binary', pos_label=1, zero_division=0)

print(f'>>> Optimal Dev Macro-F1: {best_macro_f1:.4f} at Threshold {best_thresh:.3f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} <<<')

# Inference on Test Set
test_loader = DataLoader(CJPEDataset(test_data, tok, max_length=1024, is_test=True), batch_size=16, shuffle=False)
test_ids = []
test_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        test_probs.extend(probs)
        test_ids.extend(batch['id'])

test_probs = np.array(test_probs)
test_preds = (test_probs >= best_thresh).astype(int)

# Save raw probabilities for calibrated multi-model ensembling
np.save('/kaggle/working/test_probs.npy', test_probs)
try:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj='/kaggle/working/test_probs.npy',
        path_in_repo=f"phase4/{SUB_ZIP_NAME.replace('.zip', '_probs.npy')}",
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    print('Uploaded raw test probabilities to HF!')
except Exception as e:
    print('Prob upload exception:', e)

# Package into official format: [{"id": "...", "pred": 0|1}, ...]
final_predictions = [{'id': str(did), 'pred': int(p)} for did, p in zip(test_ids, test_preds)]
assert len(final_predictions) == len(test_data), f'Test count mismatch: {len(final_predictions)} vs {len(test_data)}'
print(f'✅ Exact match on {len(final_predictions)} test cases verified!')

# Save files
pred_path = '/kaggle/working/prediction.json'
zip_path = f'/kaggle/working/{SUB_ZIP_NAME}'

with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(final_predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')

print(f'Created {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')

# Upload to Hugging Face
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo=f'phase4/{SUB_ZIP_NAME}',
    repo_id=TARGET_REPO,
    repo_type='dataset'
)
print('Successfully uploaded Phase 4 submission to Hugging Face!')
